# School Districting Optimization with Gurobi

This notebook presents a fully synthetic school districting problem formulated as a mixed-integer linear programming model.

The goal is to assign neighborhood-grade student groups to eligible schools while minimizing total student-weighted travel distance and respecting school capacities.

All schools, neighborhoods, names, and numerical values in this notebook are fictional and created only for educational purposes.

## Learning Objectives

By the end of this notebook, you should be able to:

- define sets, parameters, and binary decision variables for a districting problem,
- formulate an assignment objective,
- enforce school capacity and grade eligibility constraints,
- implement the model with the Gurobi Python API,
- interpret assignment and capacity-utilization results,
- extend the model with a continuity preference.

## 1. Problem Scenario

A fictional education region contains six residential neighborhoods and three schools. Each neighborhood contains students in kindergarten and grade 1.

The district must assign every neighborhood-grade group to exactly one school. A group cannot be split between multiple schools.

The assignment plan should satisfy three basic requirements:

1. Every neighborhood-grade group must be assigned.
2. No school-grade capacity may be exceeded.
3. Students may only be assigned to schools that offer their grade.

The baseline objective minimizes total student-weighted travel distance.

## 2. Mathematical Formulation

### Sets and Indices

$$N = \text{set of neighborhoods, indexed by } n$$

$$S = \text{set of schools, indexed by } s$$

$$G = \text{set of grades, indexed by } g$$

### Parameters

$$P_{n,g} = \text{number of students in neighborhood } n \text{ and grade } g$$

$$D_{n,s} = \text{distance from neighborhood } n \text{ to school } s$$

$$C_{s,g} = \text{capacity of school } s \text{ for grade } g$$

$$A_{s,g} =
\begin{cases}
1, & \text{if school } s \text{ serves grade } g\\
0, & \text{otherwise}
\end{cases}$$

### Decision Variable

$$x_{n,s,g} =
\begin{cases}
1, & \text{if neighborhood } n \text{ in grade } g \text{ is assigned to school } s\\
0, & \text{otherwise}
\end{cases}$$

### Objective Function

$$\min Z = \sum_{n \in N}\sum_{s \in S}\sum_{g \in G}
D_{n,s}P_{n,g}x_{n,s,g}$$

### Assignment Constraints

$$\sum_{s \in S}x_{n,s,g}=1
\qquad \forall n \in N,\; g \in G$$

### Capacity Constraints

$$\sum_{n \in N}P_{n,g}x_{n,s,g}\leq C_{s,g}
\qquad \forall s \in S,\; g \in G$$

### Grade Eligibility Constraints

$$x_{n,s,g}\leq A_{s,g}
\qquad \forall n \in N,\; s \in S,\; g \in G$$

### Binary Restrictions

$$x_{n,s,g}\in\{0,1\}
\qquad \forall n \in N,\; s \in S,\; g \in G$$

## 3. Import Gurobi

In [ ]:
from gurobipy import GRB, Model, quicksum

## 4. Synthetic Data

In [ ]:
students = {
    ("Northside", "K"): 20,
    ("Northside", "1"): 18,
    ("Lakeside", "K"): 22,
    ("Lakeside", "1"): 20,
    ("Hillview", "K"): 25,
    ("Hillview", "1"): 21,
    ("Riverside", "K"): 18,
    ("Riverside", "1"): 17,
    ("Westfield", "K"): 24,
    ("Westfield", "1"): 23,
    ("Eastgate", "K"): 21,
    ("Eastgate", "1"): 19,
}

capacities = {
    ("School_A", "K"): 50,
    ("School_A", "1"): 50,
    ("School_B", "K"): 50,
    ("School_B", "1"): 50,
    ("School_C", "K"): 50,
    ("School_C", "1"): 50,
}

schools_serving_grades = {
    "School_A": ["K", "1"],
    "School_B": ["K", "1"],
    "School_C": ["K", "1"],
}

distances = {
    ("Northside", "School_A"): 1.2,
    ("Northside", "School_B"): 3.8,
    ("Northside", "School_C"): 5.0,
    ("Lakeside", "School_A"): 1.8,
    ("Lakeside", "School_B"): 2.6,
    ("Lakeside", "School_C"): 4.2,
    ("Hillview", "School_A"): 3.4,
    ("Hillview", "School_B"): 1.4,
    ("Hillview", "School_C"): 3.0,
    ("Riverside", "School_A"): 4.0,
    ("Riverside", "School_B"): 1.6,
    ("Riverside", "School_C"): 2.2,
    ("Westfield", "School_A"): 5.2,
    ("Westfield", "School_B"): 3.0,
    ("Westfield", "School_C"): 1.3,
    ("Eastgate", "School_A"): 4.6,
    ("Eastgate", "School_B"): 2.8,
    ("Eastgate", "School_C"): 1.5,
}

neighborhoods = sorted({n for n, _ in students})
schools = sorted(schools_serving_grades)
grades = sorted({g for _, g in students})

print("Neighborhoods:", neighborhoods)
print("Schools:", schools)
print("Grades:", grades)

## 5. Feasible Assignment Index

Instead of creating variables for impossible assignments and later forcing them to zero, the model creates binary variables only for valid neighborhood-school-grade combinations.

In [ ]:
feasible_assignments = [
    (n, s, g)
    for n in neighborhoods
    for s in schools
    for g in grades
    if (n, g) in students
    and g in schools_serving_grades[s]
    and (s, g) in capacities
    and (n, s) in distances
]

print(f"Number of feasible assignment variables: {len(feasible_assignments)}")

## 6. Build the Baseline MILP

In [ ]:
model = Model("SchoolDistrictingCourse")

x = model.addVars(
    feasible_assignments,
    vtype=GRB.BINARY,
    name="assign",
)

model.setObjective(
    quicksum(
        distances[n, s] * students[n, g] * x[n, s, g]
        for n, s, g in feasible_assignments
    ),
    GRB.MINIMIZE,
)

for n in neighborhoods:
    for g in grades:
        if (n, g) in students:
            eligible_schools = [
                s for s in schools if (n, s, g) in x
            ]
            model.addConstr(
                quicksum(x[n, s, g] for s in eligible_schools) == 1,
                name=f"assignment_{n}_{g}",
            )

for s in schools:
    for g in grades:
        if (s, g) in capacities:
            model.addConstr(
                quicksum(
                    students[n, g] * x[n, s, g]
                    for n in neighborhoods
                    if (n, s, g) in x
                )
                <= capacities[s, g],
                name=f"capacity_{s}_{g}",
            )

## 7. Solve the Baseline Model

In [ ]:
model.optimize()

if model.Status == GRB.OPTIMAL:
    print(f"Optimal weighted travel distance: {model.ObjVal:.2f}")
elif model.Status == GRB.INFEASIBLE:
    model.computeIIS()
    model.write("school_districting_course.ilp")
    raise RuntimeError(
        "The model is infeasible. An IIS was written to school_districting_course.ilp."
    )
else:
    raise RuntimeError(f"Optimization ended with status code {model.Status}.")

## 8. Assignment Results

In [ ]:
assignment_rows = []

for n, s, g in feasible_assignments:
    if x[n, s, g].X > 0.5:
        assignment_rows.append(
            {
                "Neighborhood": n,
                "Grade": g,
                "School": s,
                "Students": students[n, g],
                "Distance": distances[n, s],
                "WeightedDistance": students[n, g] * distances[n, s],
            }
        )

for row in sorted(assignment_rows, key=lambda r: (r["Neighborhood"], r["Grade"])):
    print(
        f'{row["Neighborhood"]:10s} | Grade {row["Grade"]:>1s} | '
        f'{row["School"]:8s} | Students {row["Students"]:2d} | '
        f'Distance {row["Distance"]:.1f}'
    )

## 9. School Capacity Utilization

In [ ]:
for s in schools:
    for g in grades:
        if (s, g) not in capacities:
            continue

        enrollment = sum(
            students[n, g] * x[n, s, g].X
            for n in neighborhoods
            if (n, s, g) in x
        )

        capacity = capacities[s, g]
        utilization = 100.0 * enrollment / capacity

        print(
            f"{s} | Grade {g} | "
            f"Enrollment {enrollment:.0f}/{capacity} | "
            f"Utilization {utilization:.1f}%"
        )

## 10. Extension: Continuity Preference

Travel distance is not always the only planning objective. A district may also prefer to keep a neighborhood-grade group at its previous school when capacity allows.

Define:

$$H_{n,s,g} =
\begin{cases}
1, & \text{if school } s \text{ was the previous school for neighborhood } n \text{ and grade } g\\
0, & \text{otherwise}
\end{cases}$$

A penalty can be added when a group moves away from its previous school.

Let:

$$\lambda = \text{continuity penalty per reassigned student}$$

The extended objective is:

$$\min Z =
\sum_{n \in N}\sum_{s \in S}\sum_{g \in G}
D_{n,s}P_{n,g}x_{n,s,g}
+
\lambda
\sum_{n \in N}\sum_{g \in G}
P_{n,g}
\left(
1-\sum_{s \in S}H_{n,s,g}x_{n,s,g}
\right)$$

A larger value of $\lambda$ places more importance on keeping existing assignments.

## 11. Build and Solve the Continuity Model

In [ ]:
previous_school = {
    ("Northside", "K"): "School_A",
    ("Northside", "1"): "School_A",
    ("Lakeside", "K"): "School_A",
    ("Lakeside", "1"): "School_A",
    ("Hillview", "K"): "School_B",
    ("Hillview", "1"): "School_B",
    ("Riverside", "K"): "School_B",
    ("Riverside", "1"): "School_B",
    ("Westfield", "K"): "School_C",
    ("Westfield", "1"): "School_C",
    ("Eastgate", "K"): "School_B",
    ("Eastgate", "1"): "School_B",
}

continuity_penalty = 1.0

continuity_model = Model("SchoolDistrictingWithContinuity")

xc = continuity_model.addVars(
    feasible_assignments,
    vtype=GRB.BINARY,
    name="assign",
)

travel_cost = quicksum(
    distances[n, s] * students[n, g] * xc[n, s, g]
    for n, s, g in feasible_assignments
)

reassignment_cost = quicksum(
    continuity_penalty
    * students[n, g]
    * (
        1
        - quicksum(
            xc[n, s, g]
            for s in schools
            if (n, s, g) in xc and previous_school[n, g] == s
        )
    )
    for n in neighborhoods
    for g in grades
    if (n, g) in students
)

continuity_model.setObjective(
    travel_cost + reassignment_cost,
    GRB.MINIMIZE,
)

for n in neighborhoods:
    for g in grades:
        if (n, g) in students:
            continuity_model.addConstr(
                quicksum(
                    xc[n, s, g]
                    for s in schools
                    if (n, s, g) in xc
                )
                == 1,
                name=f"assignment_{n}_{g}",
            )

for s in schools:
    for g in grades:
        if (s, g) in capacities:
            continuity_model.addConstr(
                quicksum(
                    students[n, g] * xc[n, s, g]
                    for n in neighborhoods
                    if (n, s, g) in xc
                )
                <= capacities[s, g],
                name=f"capacity_{s}_{g}",
            )

continuity_model.optimize()

if continuity_model.Status != GRB.OPTIMAL:
    raise RuntimeError(
        f"Optimization ended with status code {continuity_model.Status}."
    )

print(f"Extended objective value: {continuity_model.ObjVal:.2f}")

## 12. Compare New and Previous Assignments

In [ ]:
moved_students = 0

for n in neighborhoods:
    for g in grades:
        if (n, g) not in students:
            continue

        selected_school = next(
            s
            for s in schools
            if (n, s, g) in xc and xc[n, s, g].X > 0.5
        )

        previous = previous_school[n, g]
        moved = selected_school != previous

        if moved:
            moved_students += students[n, g]

        print(
            f"{n:10s} | Grade {g} | "
            f"Previous {previous:8s} | New {selected_school:8s} | "
            f"Moved {moved}"
        )

print(f"Total students reassigned: {moved_students}")

## 13. Discussion

The baseline model minimizes travel distance subject to capacity and grade eligibility.

The continuity model adds a second planning concern: avoiding unnecessary reassignment. The continuity penalty controls the trade-off. When the penalty is small, travel distance dominates. When the penalty is large, the model is more willing to accept longer travel in order to preserve previous assignments.

This is a standard modeling pattern in mixed-integer programming: a primary operational cost is combined with a penalty representing a secondary planning preference.

## 14. Exercises

1. Change one school's kindergarten capacity and solve the baseline model again.
2. Increase `continuity_penalty` from `1.0` to `3.0`. Compare the assignments.
3. Reduce one school-grade capacity until the model must move a neighborhood-grade group.
4. Add a fourth fictional school and define its capacities and distances.
5. Add grade 2 to the data and extend all required parameters.
6. Add a maximum acceptable travel distance rule.
7. Add a target enrollment penalty so that schools are encouraged to operate near a desired enrollment level.

## License Notice

This educational material is provided under the repository's non-commercial license.

Commercial use, resale, paid redistribution, or incorporation into a commercial product or service is prohibited. See the repository `LICENSE` file for the complete terms.